# 01 · Fine-tuning v1 — YOLO26s para placas brasileiras (Google Colab)

Este notebook treina o modelo base `number-plate-yolo26s.pt` no dataset **UFPR-ALPR**
com hiperparâmetros escolhidos manualmente. O resultado é o modelo **`best_placas_v1.pt`**.

É a versão em células do script `src/treinamento/fine_tuning_colab.py`, que gerou o v1 em 05/05/2026.
Os hiperparâmetros são exatamente os mesmos.

## Requisitos

- **Google Colab com GPU T4**: *Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU*
- **Dataset UFPR-ALPR**: uso restrito a pesquisa. O acesso deve ser solicitado à UFPR:
  https://web.inf.ufpr.br/vri/databases/ufpr-alpr/
- **Modelo base** `number-plate-yolo26s.pt` (Muhammad Rizwan Munawar, AGPL-3.0)
- No Google Drive, uma pasta `placas-yolo` organizada assim:

```
Meu Drive/placas-yolo/
├── number-plate-yolo26s.pt
└── UFPR - Dataset/
    └── yj4Iu2-UFPR-ALPR.zip
```

## 1) Montar o Google Drive

Conecta o Colab ao Google Drive para ler o modelo base e o dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Instalar dependências

In [ ]:
!pip install ultralytics -q

## 3) Configurações

Caminhos dos arquivos no Drive e hiperparâmetros do treino.

In [ ]:
from pathlib import Path

# Arquivos no Drive
PASTA_PROJETO  = Path("/content/drive/MyDrive/placas-yolo")
CAMINHO_MODELO = PASTA_PROJETO / "number-plate-yolo26s.pt"
CAMINHO_ZIP    = PASTA_PROJETO / "UFPR - Dataset" / "yj4Iu2-UFPR-ALPR.zip"

# Pastas de trabalho no disco local do Colab (mais rápido que ler do Drive)
PASTA_UFPR   = Path("/content/UFPR-ALPR dataset")
PASTA_YOLO   = Path("/content/dataset")
CAMINHO_YAML = Path("/content/dataset.yaml")

# Hiperparâmetros do v1 (escolhidos manualmente)
EPOCHS      = 50      # voltas completas no dataset de treino
BATCH       = 8       # imagens por passo (limite de memória da T4 com imgsz 1280)
IMGSZ       = 1280    # resolução maior = mais detalhe para placas pequenas (motos)
LR0         = 0.001   # learning rate inicial, 10x menor que o do treino original
PATIENCE    = 10      # para o treino se não melhorar em 10 épocas seguidas
FREEZE      = 10      # congela as 10 primeiras camadas (backbone)
DEGREES     = 10      # rotação aleatória de até ±10°
PERSPECTIVE = 0.001   # distorção de perspectiva

for arquivo in (CAMINHO_MODELO, CAMINHO_ZIP):
    print(("OK      " if arquivo.exists() else "FALTANDO") + f"  {arquivo}")

## 4) Extrair o dataset

Descompacta o zip do Drive para o disco local do Colab (~9,7 GB, leva alguns minutos).

In [ ]:
import zipfile

if not PASTA_UFPR.exists():
    print("Extraindo...")
    with zipfile.ZipFile(CAMINHO_ZIP, "r") as zf:
        zf.extractall(PASTA_UFPR.parent)
print(f"Pronto: {PASTA_UFPR}")

## 5) Converter as anotações para o formato YOLO

Cada imagem do UFPR tem um `.txt` com os 4 cantos da placa (linha `corners:`).
O YOLO espera `classe cx cy largura altura`, com valores de 0 a 1.
As imagens entram como atalho (*symlink*) para não duplicar 9,7 GB.

In [ ]:
import os

LARGURA, ALTURA = 1920, 1080   # resolução fixa das imagens do UFPR
splits = {"training": "train", "validation": "val", "testing": "test"}

for split_ufpr, split_yolo in splits.items():
    pasta_imgs = PASTA_YOLO / "images" / split_yolo
    pasta_lbls = PASTA_YOLO / "labels" / split_yolo
    pasta_imgs.mkdir(parents=True, exist_ok=True)
    pasta_lbls.mkdir(parents=True, exist_ok=True)

    contagem = 0
    for track in sorted((PASTA_UFPR / split_ufpr).iterdir()):
        if not track.is_dir():
            continue
        for png in sorted(track.glob("*.png")):
            txt = png.with_suffix(".txt")
            if not txt.exists():
                continue

            destino_img = pasta_imgs / png.name
            if not destino_img.exists():
                os.symlink(png, destino_img)

            label = ""
            with open(txt, encoding="utf-8") as f:
                for linha in f:
                    if linha.startswith("corners:"):
                        pares = linha.replace("corners:", "").strip().split()
                        xs = [int(p.split(",")[0]) for p in pares]
                        ys = [int(p.split(",")[1]) for p in pares]
                        cx = (min(xs) + max(xs)) / 2 / LARGURA
                        cy = (min(ys) + max(ys)) / 2 / ALTURA
                        w  = (max(xs) - min(xs)) / LARGURA
                        h  = (max(ys) - min(ys)) / ALTURA
                        label = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n"
                        break

            (pasta_lbls / (png.stem + ".txt")).write_text(label)
            contagem += 1

    print(f"  [{split_ufpr:>10}] -> {contagem} imagens")

## 6) Gerar o `dataset.yaml`

Diz ao YOLO onde estão as imagens e qual é a classe (uma só: `license_plate`).

In [ ]:
CAMINHO_YAML.write_text(f"""path: {PASTA_YOLO}
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: license_plate
""")
print(CAMINHO_YAML.read_text())

## 7) Fine-tuning

Treino com GPU T4. Estimativa: algumas horas (o early stopping pode encerrar antes das 50 épocas).

In [ ]:
from ultralytics import YOLO

modelo = YOLO(str(CAMINHO_MODELO))

modelo.train(
    data=str(CAMINHO_YAML),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    lr0=LR0,
    cos_lr=True,            # learning rate decai em curva de cosseno
    patience=PATIENCE,
    freeze=FREEZE,
    degrees=DEGREES,
    perspective=PERSPECTIVE,
    project="/content/runs/fine_tuning",
    name="placas_brasileiras_colab_v1",
    exist_ok=True,
)

## 8) Salvar o modelo treinado no Drive

O melhor resultado do treino fica em `weights/best.pt`, dentro do Colab, e é apagado quando a sessão fecha.
Por isso ele é copiado para o Drive.

In [ ]:
import shutil

origem  = "/content/runs/fine_tuning/placas_brasileiras_colab_v1/weights/best.pt"
destino = PASTA_PROJETO / "best_placas_v1.pt"

shutil.copy(origem, destino)
print(f"Salvo em {destino}")